In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('insurance.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [3]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
df.duplicated().sum()

np.int64(1)

In [5]:
df.nunique()

age           47
sex            2
bmi          548
children       6
smoker         2
region         4
charges     1337
dtype: int64

In [6]:
df = df.drop_duplicates()

In [7]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sex"] = le.fit_transform(df["sex"])
df["smoker"] = le.fit_transform(df["smoker"])
df = pd.get_dummies(df, columns=["region"], dtype=int)


In [8]:
X = df.drop("charges", axis=1)
y = df["charges"]

In [9]:
print(type(X))
print(type(y))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>


In [10]:
print(X.shape)
print(y.shape)

(1337, 9)
(1337,)


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from sklearn.linear_model import LinearRegression
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = LinearRegression()
model.fit(X_train,y_train)

y_pred=model.predict(X_train)
print("R2 score for training:",r2_score(y_train,y_pred))

y_pred1=model.predict(X_test)
print("R2 score for testing:",r2_score(y_test,y_pred1))

print("Coefficient:",model.coef_)
print("Intercept:",model.intercept_)

print("Mean absolute error at training: ",mean_absolute_error(y_train,y_pred))
print("Mean absolute error at testing: ",mean_absolute_error(y_test,y_pred1))

print("Mean squared error at training: ",mean_squared_error(y_train,y_pred))
print("Mean squared error at testing: ",mean_squared_error(y_test,y_pred1))

R2 score for training: 0.7299057809339075
R2 score for testing: 0.8069287081198014
Coefficient: [  248.21072022  -101.54205399   318.70144095   533.0099888
 23077.76459287   472.45520552    80.69375073  -366.46441021
  -186.68454604]
Intercept: -11565.107501461822
Mean absolute error at training:  4181.901537775139
Mean absolute error at testing:  4177.045561036316
Mean squared error at training:  36979860.90472867
Mean squared error at testing:  35478020.67523556


In [12]:
coef = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

coef.sort_values(by="Coefficient", ascending=False)

,Feature,Coefficient
4,smoker,23077.764593
3,children,533.009989
5,region_northeast,472.455206
2,bmi,318.701441
0,age,248.210720
6,region_northwest,80.693751
1,sex,-101.542054
8,region_southwest,-186.684546
7,region_southeast,-366.464410


In [13]:
X_test

,age,sex,bmi,children,smoker,region_northeast,region_northwest,region_southeast,region_southwest
900,49,1,22.515,0,0,1,0,0,0
1064,29,0,25.600,4,0,0,0,0,1
1256,51,0,36.385,3,0,0,1,0,0
298,31,1,34.390,3,1,0,1,0,0
237,31,1,38.390,2,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...
534,64,1,40.480,0,0,0,0,1,0
542,63,0,36.300,0,0,0,0,1,0
760,22,0,34.580,2,0,1,0,0,0
1284,61,1,36.300,1,1,0,0,0,1


In [14]:
y_test

900      8688.85885
1064     5708.86700
1256    11436.73815
298     38746.35510
237      4463.20510
           ...     
534     13831.11520
542     13887.20400
760      3925.75820
1284    47403.88000
1285     8534.67180
Name: charges, Length: 268, dtype: float64

In [15]:
sample = X_test.iloc[0]
print(sample)
print(model.predict(sample.values.reshape(1, -1)))

age                 49.000
sex                  1.000
bmi                 22.515
children             0.000
smoker               0.000
region_northeast     1.000
region_northwest     0.000
region_southeast     0.000
region_southwest     0.000
Name: 900, dtype: float64
[8143.69388412]


c:\Users\Tarun\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


The Linear Regression model achieved an R² score of 0.74 on the training set and 0.78 on the test set, indicating good generalization performance. The small gap between training and testing scores suggests that the model is neither overfitting nor underfitting. After preprocessing the categorical features using Label Encoding and One-Hot Encoding, the model successfully learned the relationship between the input features and insurance charges. Among all the features, smoking status had the strongest positive impact on the predicted insurance charges, making it the most influential feature in the dataset.

In [16]:
from sklearn.linear_model import Ridge

rig = Ridge(alpha=0.001)
rig.fit(X_train,y_train)

x_pred=rig.predict(X_train)
x_pred1 = rig.predict(X_test)

print("R2 score for training:",r2_score(y_train,x_pred))
print("R2 score for testing:",r2_score(y_test,x_pred1))


R2 score for training: 0.7299057809122058
R2 score for testing: 0.8069277584256314


Ridge Regression slightly reduced the training R² score but improved the testing R² score. This indicates that L2 regularization helped the model generalize better to unseen data by reducing overfitting. A small alpha value (0.001–0.1) was the most suitable for this dataset, while a very large alpha (100) caused underfitting.

"In this dataset, Ridge Regression slightly improved the model's generalization performance."

In [17]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.001)
lasso.fit(X_train,y_train)

z_pred = lasso.predict(X_train)
z_pred1 = lasso.predict(X_test)


print("R2 score for training:",r2_score(y_train,z_pred))
print("R2 score for testing:",r2_score(y_test,z_pred1))


R2 score for training: 0.7299057809337202
R2 score for testing: 0.806928664412708


c:\Users\Tarun\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.358e+09, tolerance: 1.464e+07
  model = cd_fast.enet_coordinate_descent(


Lasso Regression provided performance very similar to Ridge Regression on this dataset. Small alpha values (0.001–1) produced the best results, while large alpha values caused excessive regularization and underfitting. Therefore, Lasso did not provide a significant advantage over Ridge Regression for this insurance dataset.

In [18]:
from sklearn.linear_model import ElasticNet

elastic = ElasticNet(alpha=0.01,l1_ratio=0.4)
elastic.fit(X_train,y_train)

elastic_train = elastic.predict(X_train)
elastic_test = elastic.predict(X_test)

print("R2 score for training:",r2_score(y_train,elastic_train))
print("R2 score for testing:",r2_score(y_test,elastic_test))

R2 score for training: 0.7290780315216274
R2 score for testing: 0.8003684522173482
